# Trip Explorer — Hackathon Dataset Walkthrough

Visualize one trip's content to understand the dataset structure: road frames, driver state, TTC ground truth, behavior events.

*(Notebook khám phá 1 trip — giúp bạn hiểu cấu trúc dataset: ảnh camera, trạng thái tài xế, TTC ground truth, các sự kiện hành vi lái xe.)*

**Cách chạy notebook này — chọn 1 trong 2:**

1. **Local (máy cá nhân):** đảm bảo bạn đang mở notebook này từ đúng vị trí
   `package_starterkit/team_kit/explore_trip.ipynb` và đã có `package_starterkit/data/T01-Sample/...`.
   Cài thư viện rồi chạy thẳng xuống **Cell 1**:
   ```bash
   pip install opencv-python numpy pandas matplotlib pillow
   ```
2. **Google Colab:** chạy **Cell 0 (Colab setup)** ngay bên dưới trước — cell đó sẽ mount Drive,
   giải nén `team_kit.zip` + trip zip đúng cách, và cài `opencv-python-headless`. Sau đó chạy
   tiếp xuống Cell 1 như bình thường (không cần sửa tay `TRIP_DIR`).

Set `TRIP_DIR` to point at one generated trip folder (e.g. `./data/T01-Sample`).

*(Đặt biến `TRIP_DIR` trỏ đến 1 thư mục trip đã sinh ra, ví dụ `./data/T01-Sample`.)*

> **Lưu ý quan trọng:** 6 trip luyện tập (**T01-Sample .. T06-Sample**) có đầy đủ ground truth (GT) để bạn tự kiểm tra —
> 10 trip chấm điểm (**T01d .. T10d**) đã bị ẩn GT (dùng để chấm điểm). Nếu bạn đổi `TRIP_DIR` sang 1 trong 10 trip đó,
> một số cell bên dưới sẽ hiển thị dữ liệu rỗng/mặc định thay vì lỗi — đây là chủ đích, không phải bug.


## Cell 0 — Colab setup (bỏ qua nếu chạy local)

*(Chỉ cần chạy cell code ngay bên dưới nếu bạn đang mở notebook này trên **Google Colab**.
Nếu chạy local (VS Code / Jupyter trên máy) thì bỏ qua, chuyển thẳng xuống "Cell 1: Setup".)*

Trước khi chạy, sửa 2 biến `TEAM_KIT_ZIP` và `TRIP_ZIPS` bên dưới cho khớp với đường dẫn thật
trong Google Drive của bạn (xem `HUONG_DAN_NGUOI_MOI.md` mục "Cách 2" nếu chưa upload zip lên Drive).

> **Vì sao cần cell này:** lỗi phổ biến nhất khi chạy trên Colab là
> `ModuleNotFoundError: No module named 'team_kit'`. Lỗi này xảy ra khi ai đó giải nén xong rồi
> lỡ tay "làm phẳng" thư mục (ví dụ `mv team_kit/* .` rồi xoá thư mục `team_kit`) — cell dưới đây
> **giữ nguyên** thư mục con `team_kit/` để câu lệnh `from team_kit.dataset_loader import TripDataset`
> ở Cell 1 luôn import được, trên cả Colab lẫn local.

In [ ]:
# ===== Cell 0: Colab setup — mount Drive + giải nén ĐÚNG CÁCH (bỏ qua nếu chạy local) =====
# Cell này TỰ BỎ QUA nếu không phát hiện đang chạy trên Colab, nên chạy vô hại kể cả khi
# bạn đang chạy local (không cần xoá/comment cell này đi).
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # >>> SỬA 2 BIẾN DƯỚI ĐÂY cho khớp với đường dẫn thật trong Drive của bạn <<<
    TEAM_KIT_ZIP = '/content/drive/MyDrive/hackathon/team_kit.zip'
    TRIP_ZIPS = [
        '/content/drive/MyDrive/hackathon/T01-Sample.zip',
        # Thêm dòng khác nếu muốn giải nén thêm trip, ví dụ:
        # '/content/drive/MyDrive/hackathon/T02-Sample.zip',
    ]

    import os
    os.makedirs('/content/project/data', exist_ok=True)

    # Giải nén team_kit.zip -> /content/project/team_kit/ (GIỮ NGUYÊN thư mục con team_kit,
    # KHÔNG mv/flatten nội dung ra ngoài — nếu không "from team_kit.dataset_loader import" sẽ lỗi)
    if not os.path.exists('/content/project/team_kit/dataset_loader.py'):
        !unzip -q "{TEAM_KIT_ZIP}" -d /content/project
    else:
        print('team_kit đã có sẵn trong /content/project — bỏ qua giải nén.')

    for trip_zip in TRIP_ZIPS:
        !unzip -q -o "{trip_zip}" -d /content/project/data

    # opencv-python-headless: bản không cần giao diện đồ hoạ, tránh lỗi "libGL.so.1" trên Colab
    !pip install -q opencv-python-headless

    assert os.path.exists('/content/project/team_kit/dataset_loader.py'), (
        "Không tìm thấy /content/project/team_kit/dataset_loader.py sau khi giải nén — "
        "kiểm tra lại đường dẫn TEAM_KIT_ZIP ở trên."
    )
    print('Xong. Cell 1 bên dưới sẽ tự nhận diện PROJECT_ROOT = /content/project')
else:
    print('Không chạy trên Colab — bỏ qua Cell 0, chuyển xuống Cell 1.')


In [ ]:
# ===== Cell 1: Setup — nạp thư viện + load 1 trip =====
# Cell này làm 3 việc:
#   1. Tự dò PROJECT_ROOT (thư mục chứa team_kit/dataset_loader.py) rồi thêm vào sys.path
#      để import được `team_kit`, bất kể bạn chạy local hay đã giải nén qua Cell 0 trên Colab.
#   2. Import các thư viện xử lý ảnh/số liệu/vẽ biểu đồ
#   3. Load 1 trip cụ thể qua TripDataset (đọc JSON + biết đường dẫn ảnh/driver/depth)

import sys
from pathlib import Path

# Thử lần lượt vài vị trí quen thuộc: cwd hiện tại (nếu mở notebook ngay trong team_kit/),
# thư mục cha của cwd, và đường dẫn chuẩn của Colab sau khi chạy Cell 0.
_candidates = [Path.cwd(), Path.cwd().parent, Path('/content/project')]
PROJECT_ROOT = next((c for c in _candidates if (c / 'team_kit' / 'dataset_loader.py').exists()), None)

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Khong tim thay team_kit/dataset_loader.py o cac vi tri quen thuoc: "
        f"{[str(c) for c in _candidates]}.\n"
        "- Neu chay LOCAL: mo notebook nay tu trong thu muc package_starterkit/team_kit/.\n"
        "- Neu chay tren Google Colab: chay 'Cell 0 - Colab setup' o phia tren truoc, va dam bao "
        "KHONG lam phang (mv) thu muc con 'team_kit' sau khi giai nen."
    )

sys.path.insert(0, str(PROJECT_ROOT))
print(f'PROJECT_ROOT = {PROJECT_ROOT}')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from team_kit.dataset_loader import TripDataset

# ĐỔI dòng dưới đây để trỏ tới trip bạn muốn xem.
# Khuyên dùng T01-Sample..T06-Sample lúc mới bắt đầu (có đủ GT để đối chiếu).
TRIP_DIR = PROJECT_ROOT / 'data' / 'T01-Sample'

print(f'Loading trip: {TRIP_DIR}')
ds = TripDataset(TRIP_DIR)
print(f'Frames: {len(ds)}, duration: {ds.metadata["duration_sec"]}s @ {ds.metadata["fps"]} FPS')


## 1. Trip overview

*(Tổng quan trip: map, thời lượng, điểm số, thống kê driver state...)*

In [ ]:
# ===== Cell 3: In tổng quan trip =====
# ds.summary() trả về 1 dict tóm tắt: trip_id, map, số frame, safe_driving_score...
# Với 10 trip chấm điểm (T0Xd, bị redact), các field liên quan GT (safe_driving_score, driver_subject...)
# sẽ là None — đó là chủ đích (không phải lỗi), vì dữ liệu đó bị ẩn để chấm điểm.
import json
print(json.dumps(ds.summary(), indent=2, default=str))


In [ ]:
# ===== Cell 4: Danh sách event đã xảy ra trong trip =====
# ds.events_log lấy trực tiếp từ JSON (events_log không bị redact hoàn toàn —
# chỉ "params" chi tiết bị xóa ở 10 trip chấm điểm (T0Xd), còn "type" và "t" (thời điểm) vẫn giữ lại
# vì đó là hint nhẹ chấp nhận được, giống cách hệ thống ADAS thật cũng có cảnh báo trước).
print(f'Events: {len(ds.events_log)} fired')
for ev in ds.events_log:
    print(f'  {ev}')


## 2. Driver state timeline

Shows how driver state evolves over the trip duration (5 NTHU-sourced states).

*(Biểu đồ trạng thái tài xế theo thời gian — 5 trạng thái: alert/drowsy/yawning/distracted/microsleep.
Chỉ có dữ liệu thật ở 6 trip full-GT (T01-Sample..T06-Sample); ở 10 trip bị redact (T0Xd) sẽ hiện "unknown" cho toàn bộ trip.)*

In [ ]:
# ===== Cell 6: Vẽ timeline trạng thái tài xế =====
# Mỗi state được vẽ thành 1 dải màu riêng theo trục thời gian (dùng scatter với marker '|'
# để tạo hiệu ứng "vạch kẻ" liên tục cho mỗi state).
df = ds.frames_df   # DataFrame phẳng, mỗi dòng = 1 frame (xem dataset_loader.py để biết đủ cột)

STATE_COLORS = {
    'alert':      '#2ecc71',   # xanh lá — tỉnh táo
    'drowsy':     '#f39c12',   # cam — buồn ngủ
    'yawning':    '#3498db',   # xanh dương — đang ngáp
    'distracted': '#9b59b6',   # tím — mất tập trung
    'microsleep': '#e74c3c',   # đỏ — ngủ gật (nguy hiểm nhất)
}

fig, ax = plt.subplots(figsize=(14, 2.5))
for state, color in STATE_COLORS.items():
    mask = df['driver_state'] == state
    if mask.any():
        ax.scatter(df.loc[mask, 'timestamp'], [1]*int(mask.sum()), 
                   c=color, s=12, label=state, marker='|')
ax.set_yticks([])
ax.set_xlabel('Time (s)')
ax.set_title(f'Driver state timeline — {ds.trip_id}')
ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=9)
ax.set_xlim(0, ds.metadata['duration_sec'])
plt.tight_layout()
plt.show()


In [ ]:
# ===== Cell 7: Alertness score (dạng liên tục, chi tiết hơn discrete state) =====
# alertness_score: 0.0 = microsleep (nguy hiểm nhất) -> 1.0 = hoàn toàn tỉnh táo
# Đường kẻ ngang 0.5 là ngưỡng tham khảo "impairment" (không phải ngưỡng chính thức của hệ thống).
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(df['timestamp'], df['alertness_score'], color='#27ae60', linewidth=1.5)
ax.fill_between(df['timestamp'], 0, df['alertness_score'], alpha=0.3, color='#2ecc71')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Impairment threshold')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Alertness (0=microsleep, 1=alert)')
ax.set_title('Alertness score')
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()


## 3. Ego dynamics over time

Speed and acceleration — useful for detecting harsh maneuvers.

*(Tốc độ & gia tốc của xe ego theo thời gian — dữ liệu này KHÔNG bị redact ở bất kỳ trip nào,
vì đây là input hợp lệ (telemetry của chính xe), không phải đáp án cần giấu.)*

In [ ]:
# ===== Cell 9: Tốc độ & gia tốc ego =====
# Đường đứt màu đỏ/cam là ngưỡng "harsh brake"/"harsh accel" dùng trong risk scoring thật
# (xem src/analytics/behavior.py) — giúp bạn hình dung lúc nào hệ thống sẽ gắn cờ harsh maneuver.
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

speed_limit = ds.metadata.get('speed_limit_kmh', 50)
axes[0].plot(df['timestamp'], df['speed_kmh'], color='#2980b9')
axes[0].axhline(speed_limit, color='red', linestyle='--', alpha=0.5, label=f'Speed limit {speed_limit} km/h')
axes[0].set_ylabel('Speed (km/h)')
axes[0].set_title('Ego speed')
axes[0].legend()

axes[1].plot(df['timestamp'], df['longitudinal_accel'], label='Longitudinal', color='#16a085')
axes[1].plot(df['timestamp'], df['lateral_accel'], label='Lateral', color='#c0392b', alpha=0.7)
axes[1].axhline(-0.4*9.81, color='red', linestyle=':', alpha=0.5, label='Harsh brake (-0.4g)')
axes[1].axhline(0.35*9.81, color='orange', linestyle=':', alpha=0.5, label='Harsh accel (+0.35g)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Accel (m/s²)')
axes[1].set_title('Ego acceleration')
axes[1].legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.show()


## 4. TTC + Risk score (ground truth)

Ground-truth TTC and composite risk score. Events overlaid as vertical lines.

*(TTC và risk score — đây chính là đáp án Challenge 1. Chỉ hiện dữ liệu thật ở 6 trip full-GT (T01-Sample..T06-Sample);
ở 10 trip bị redact (T0Xd), min_ttc/final_risk_score sẽ là giá trị mặc định (inf/0), đồ thị sẽ phẳng — đúng như thiết kế.)*

In [ ]:
# ===== Cell 11: TTC ground truth + risk score, có vạch đánh dấu event =====
# LƯU Ý: cell này lấy mốc thời gian event trực tiếp từ ds.events_log (đã có sẵn trong JSON),
# KHÔNG đọc file configs/trip_XX.yaml -- vì file YAML đó chứa đầy đủ "params" chi tiết của
# event (gap_m, target_speed_kmh...), là đúng loại dữ liệu đã bị xóa khỏi events_log cho
# 10 trip chấm điểm. Nếu code đọc thẳng YAML, sẽ vô tình lộ đúng dữ liệu đã cố tình ẩn.
# ds.events_log luôn có "type" + "t" (thời điểm) cho mọi trip (kể cả trip bị redact) nên
# đủ dùng để vẽ vạch đánh dấu mà không cần file config nào khác.
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# TTC plot — clip to 30s for visibility
ttc_clipped = df['min_ttc'].clip(upper=30)
axes[0].plot(df['timestamp'], ttc_clipped, color='#e67e22', linewidth=1.5)
axes[0].axhline(3.0, color='orange', linestyle='--', alpha=0.5, label='Critical zone (TTC<3s)')
axes[0].axhline(2.0, color='red', linestyle='--', alpha=0.5, label='Danger zone (TTC<2s)')
axes[0].fill_between(df['timestamp'], 0, 2.0, color='red', alpha=0.1)
axes[0].fill_between(df['timestamp'], 2.0, 3.0, color='orange', alpha=0.1)
axes[0].set_ylabel('Min TTC (s, clipped at 30)')
axes[0].set_title('Ground truth Time-to-Collision')
axes[0].legend(fontsize=8)

# Risk score
axes[1].plot(df['timestamp'], df['final_risk_score'], color='#c0392b', linewidth=1.5)
axes[1].fill_between(df['timestamp'], 0, df['final_risk_score'], alpha=0.2, color='#e74c3c')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Final risk score (0-100)')
axes[1].set_title(f'Composite risk score (base × driver_factor)')
axes[1].set_ylim(0, 100)

# Vẽ vạch đứng đánh dấu mỗi event, lấy trực tiếp từ ds.events_log (không đọc YAML config)
for ev in ds.events_log:
    t = ev.get('t')
    if t is None:
        continue
    for ax in axes:
        ax.axvline(t, color='purple', linestyle=':', alpha=0.6)
    axes[0].text(t, axes[0].get_ylim()[1]*0.9, ev.get('type', '?'),
                 rotation=90, fontsize=8, color='purple', va='top')

plt.tight_layout()
plt.show()


## 5. Driver state distribution

*(Tỷ lệ % thời gian ở mỗi trạng thái tài xế trong trip.)*

In [ ]:
# ===== Cell 13: Biểu đồ cột phân bố driver state =====
dist = df['driver_state'].value_counts(normalize=True) * 100
fig, ax = plt.subplots(figsize=(7, 4))
colors = [STATE_COLORS.get(s, '#888') for s in dist.index]
ax.bar(dist.index, dist.values, color=colors)
ax.set_ylabel('% of trip time')
ax.set_title(f'Driver state distribution — {ds.trip_id}')
for i, (state, pct) in enumerate(dist.items()):
    ax.text(i, pct + 1, f'{pct:.1f}%', ha='center', fontsize=10)
ax.set_ylim(0, max(dist.values) + 10)
plt.tight_layout()
plt.show()


## 6. Sample frames (road + driver side by side)

Visualize 4 frames sampled across the trip — see road context and corresponding driver state at each.

*(Xem thử 4 frame lấy mẫu dọc trip — ảnh camera đường + ảnh driver tương ứng, đặt cạnh nhau để dễ đối chiếu.)*

In [ ]:
# ===== Cell 15: Lấy mẫu 4 frame, hiển thị ảnh đường + ảnh driver song song =====
# np.linspace chọn 4 frame_id trải đều từ đầu đến cuối trip (không phải ngẫu nhiên).
n_samples = 4
sample_frame_ids = np.linspace(0, len(ds) - 1, n_samples).astype(int)

fig, axes = plt.subplots(n_samples, 2, figsize=(14, 3.5 * n_samples))

for row, fid in enumerate(sample_frame_ids):
    fr = ds[fid]
    
    # Road (left camera) — convert BGR to RGB for matplotlib
    # (OpenCV đọc ảnh theo thứ tự kênh màu BGR, matplotlib cần RGB nên phải đổi lại)
    try:
        road = cv2.cvtColor(ds.load_left(fid), cv2.COLOR_BGR2RGB)
        axes[row, 0].imshow(road)
    except FileNotFoundError:
        axes[row, 0].text(0.5, 0.5, 'no image', ha='center', va='center')
    axes[row, 0].set_title(
        f't={fr.timestamp:.1f}s  speed={fr.speed_kmh:.0f}km/h  '
        f'TTC={fr.min_ttc if np.isfinite(fr.min_ttc) else 99:.1f}s  '
        f'risk={fr.final_risk_score:.0f}'
    )
    axes[row, 0].axis('off')
    
    # Driver (composited từ DMD, tên field trong code vẫn ghi "NTHU" vì lý do lịch sử)
    try:
        driver = cv2.cvtColor(ds.load_driver(fid), cv2.COLOR_BGR2RGB)
        axes[row, 1].imshow(driver)
    except FileNotFoundError:
        axes[row, 1].text(0.5, 0.5, 'no driver image', ha='center', va='center')
    color = STATE_COLORS.get(fr.driver_state, '#888')
    axes[row, 1].set_title(
        f'state={fr.driver_state}  alertness={fr.alertness_score:.2f}  '
        f'eye={fr.eye_state}  pose={fr.head_pose}',
        color=color, fontweight='bold'
    )
    axes[row, 1].axis('off')

plt.tight_layout()
plt.show()


## 7. Behavior event spans

Visualize when harsh-brake, harsh-corner, tailgating, and speeding events occur.

*(Thời điểm xảy ra các hành vi: phanh gấp, tăng tốc gấp, cua gấp, chạy quá tốc độ, bám đuôi quá gần.
Các flag này KHÔNG bị redact ở bất kỳ trip nào -- xem README mục "Ground truth distribution policy"
để biết chính xác field nào bị ẩn ở 10 trip chấm điểm.)*

In [ ]:
# ===== Cell 17: Timeline các cờ hành vi (behavior flags) =====
# Mỗi hàng ngang là 1 loại hành vi, mỗi vạch "|" là 1 frame có hành vi đó xảy ra.
fig, ax = plt.subplots(figsize=(14, 3))

flag_colors = {
    'is_harsh_brake':  ('#e74c3c', 0),
    'is_harsh_accel':  ('#f39c12', 1),
    'is_harsh_corner': ('#9b59b6', 2),
    'is_speeding':     ('#3498db', 3),
    'is_tailgating':   ('#16a085', 4),
}

for flag, (color, y) in flag_colors.items():
    mask = df[flag]
    if mask.any():
        ax.scatter(df.loc[mask, 'timestamp'], [y] * int(mask.sum()),
                   c=color, marker='|', s=120)
    ax.text(-1.5, y, flag.replace('is_', ''), va='center', fontsize=9, ha='right')

ax.set_yticks([])
ax.set_xlabel('Time (s)')
ax.set_title('Behavior event flags')
ax.set_xlim(0, ds.metadata['duration_sec'])
ax.set_ylim(-0.5, 4.5)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Trip aggregate summary

Final Safe Driving Score + breakdown.

*(Điểm tổng kết cuối trip -- đây là đáp án Challenge 3, chỉ có ở 6 trip full-GT (T01-Sample..T06-Sample).)*

In [ ]:
# ===== Cell 19: In điểm tổng kết cuối trip =====
# QUAN TRỌNG: ds.trip_aggregate / ds.driver_summary là dict RỖNG ({}) ở 10 trip cham diem (T0Xd, bi redact)
# (đây là dữ liệu Challenge 3 -- bị ẩn để chấm điểm). Cell này kiểm tra rỗng trước khi in,
# để không bị crash (KeyError) nếu bạn đang xem 1 trip bị redact.
agg = ds.trip_aggregate
drv = ds.driver_summary

print('=' * 60)
print(f'TRIP {ds.trip_id} — FINAL SCORES')
print('=' * 60)

if not agg or not drv:
    print('Trip nay khong co ground truth aggregate (da bi an de cham diem).')
    print('6 trip T01-Sample..T06-Sample co day du du lieu o day.')
else:
    print(f'Safe Driving Score:  {agg["safe_driving_score"]} / 100')
    print(f'Risk classification: {agg["risk_classification"].upper()}')
    print(f'Driver fatigue:      {drv["fatigue_score"]} / 100')
    print()
    print('Event counts:')
    print(f'  Harsh brake:  {agg["harsh_brake_count"]}')
    print(f'  Harsh accel:  {agg["harsh_accel_count"]}')
    print(f'  Harsh corner: {agg["harsh_corner_count"]}')
    print(f'  Near misses:  {agg["near_miss_count"]}')
    print(f'  Speeding:     {agg["speeding_pct_time"]:.1f}% of trip')
    print(f'  Tailgating:   {agg["tailgating_pct_time"]:.1f}% of trip')
    print()
    print(f'Risk scores: avg={agg["avg_risk_score"]:.1f}  max={agg["max_risk_score"]:.1f}')


---

## Next steps for hackathon participants

1. **Build a TTC predictor** using only `ds.load_left(fid)` and `ds.load_right(fid)` (road camera only)
2. **Build a driver state classifier** using only `ds.load_driver(fid)` (in-cabin only)
3. **Fuse both signals** into a composite risk predictor — beat the baseline in `team_kit/baseline_ttc_predictor.py`

*(Các bước tiếp theo cho thí sinh:)*

1. **Xây dựng model dự đoán TTC** chỉ dùng `ds.load_left(fid)` / `ds.load_right(fid)` (ảnh camera đường, không dùng field GT có sẵn)
2. **Xây dựng model phân loại trạng thái tài xế** chỉ dùng `ds.load_driver(fid)` (ảnh trong cabin)
3. **Kết hợp cả 2 tín hiệu** thành 1 risk predictor tổng hợp — cố gắng vượt qua baseline trong `team_kit/baseline_ttc_predictor.py`

> **Nhắc lại quan trọng:** khi tự đánh giá model trên máy, chỉ dùng `evaluation.py` với **T01-Sample..T06-Sample**
> (6 trip có GT thật). Khi nộp bài cho 10 trip `T0Xd` còn lại, ban tổ chức sẽ tự chấm bằng dữ liệu GT riêng
> họ giữ -- `evaluation.py` **luôn** tự load GT từ thư mục tin cậy, không bao giờ tin vào bất kỳ cột
> "ground_truth" nào bạn tự ghi trong file predictions.csv của mình.